In [0]:
from pyspark.sql.functions import col, coalesce, when, split, regexp_extract, concat, lit

# Load both source tables
fotmob_df = spark.read.table("workspace.fotmob.player_stats_processed")
sofascore_df = spark.read.table("workspace.fotmob.sofascore_player_stats_processed")

# Normalize season format to integer for both dataframes
# "23/24" -> 24 (take second year only)
# "25" -> 2025 (convert 2-digit to full year)
def normalize_season(df):
    return df.withColumn(
        "season",
        when(
            col("season").cast("string").contains("/"),
            # For "25/26" format, take the second year and convert to full year (26 -> 2026)
            concat(lit("20"), split(col("season").cast("string"), "/")[1]).cast("int")
        ).otherwise(
            # For single year like "25", convert to 2025
            when(
                col("season").cast("int") < 100,
                concat(lit("20"), col("season").cast("string")).cast("int")
            ).otherwise(
                # Already 4-digit year, just cast to int
                col("season").cast("int")
            )
        )
    )

fotmob_df = normalize_season(fotmob_df)
sofascore_df = normalize_season(sofascore_df)

# Map Sofascore column names to Fotmob equivalents for semantic alignment
sofascore_to_fotmob_mapping = {
    'accurate_passes_percentage': 'pass_accuracy',
    'accurate_crosses': 'successful_crosses',
    'accurate_long_balls_percentage': 'long_ball_accuracy',
    'accurate_pass_percentage': 'pass_accuracy',
    'appearances': 'matches_played',
    'blocked_shots': 'blocked_scoring_attempt',
    'expected_assists': 'xa',
    'expected_goals': 'xg',
    'successful_dribbles': 'dribbles',
    'successful_dribbles_per_90': 'dribbles_per_90',
    'accurate_crosses_per_90': 'successful_crosses_per_90',
    'accurate_cross_percentage': 'long_ball_accuracy',
    'expected_assists_per_90': 'xa_per_90',
    'expected_goals_per_90': 'xg_per_90',
    'total_shots': 'shots'
}

print("\nMapping Sofascore columns to Fotmob equivalents:")
for sofascore_col, fotmob_col in sofascore_to_fotmob_mapping.items():
    if sofascore_col in sofascore_df.columns:
        sofascore_df = sofascore_df.withColumnRenamed(sofascore_col, fotmob_col)
        print(f"  • {sofascore_col} -> {fotmob_col}")

# After mapping, drop any Sofascore-only columns that don't align with Fotmob schema
fotmob_cols_set = set(fotmob_df.columns)
sofascore_cols_after_mapping = set(sofascore_df.columns)
columns_to_drop = sofascore_cols_after_mapping - fotmob_cols_set

if columns_to_drop:
    print(f"\nDropping {len(columns_to_drop)} Sofascore-only columns that don't align with Fotmob:")
    for col_to_drop in sorted(columns_to_drop):
        print(f"  • {col_to_drop}")
        sofascore_df = sofascore_df.drop(col_to_drop)
else:
    print("\nNo Sofascore-only columns to drop - all columns align with Fotmob schema")

print(f"\nFotmob stats: {fotmob_df.count()} rows, {len(fotmob_df.columns)} columns")
print(f"Sofascore stats: {sofascore_df.count()} rows, {len(sofascore_df.columns)} columns")

# Identify overlapping columns (excluding join keys)
join_keys = ['player_id', 'season', 'competition']
fotmob_cols = set(fotmob_df.columns)
sofascore_cols = set(sofascore_df.columns)

overlapping_cols = (fotmob_cols & sofascore_cols) - set(join_keys)
fotmob_only = fotmob_cols - sofascore_cols - set(join_keys)
sofascore_only = sofascore_cols - fotmob_cols - set(join_keys)

print(f"\nJoin keys: {join_keys}")
print(f"\nOverlapping columns ({len(overlapping_cols)}): {sorted(overlapping_cols)}")
print(f"\nFotmob-only columns ({len(fotmob_only)}): {sorted(list(fotmob_only))}")
print(f"\nSofascore-only columns ({len(sofascore_only)}): {sorted(list(sofascore_only))}")


Mapping Sofascore columns to Fotmob equivalents:
  • accurate_passes_percentage -> pass_accuracy
  • accurate_crosses -> successful_crosses
  • accurate_long_balls_percentage -> long_ball_accuracy
  • appearances -> matches_played
  • blocked_shots -> blocked_scoring_attempt
  • expected_assists -> xa
  • expected_goals -> xg
  • successful_dribbles -> dribbles
  • total_shots -> shots

Dropping 43 Sofascore-only columns that don't align with Fotmob:
  • accurate_crosses_per90
  • accurate_crosses_percentage
  • big_chances_missed
  • big_chances_missed_per90
  • blocked_shots_per90
  • count_rating
  • error_lead_to_goal
  • error_lead_to_goal_per90
  • expected_assists_per90
  • expected_goals_per90
  • goals_assists_sum
  • goals_assists_sum_per90
  • goals_conceded
  • goals_conceded_per90
  • goals_prevented
  • goals_prevented_per90
  • key_passes
  • key_passes_per90
  • outfielder_blocks
  • outfielder_blocks_per90
  • passes_to_assist
  • passes_to_assist_per90
  • penalty_fa

In [0]:
from pyspark.sql.functions import col as spark_col

# Both tables now have season as integer, no casting needed
# Rename overlapping columns in sofascore with _sofascore suffix
for col_name in overlapping_cols:
    sofascore_df = sofascore_df.withColumnRenamed(col_name, f"{col_name}_sofascore")

# Perform full outer join on player_id, season, competition
gold_df = fotmob_df.alias("fotmob").join(
    sofascore_df.alias("sofascore"),
    on=join_keys,
    how="full_outer"
)

print(f"After join: {gold_df.count()} rows")

# For overlapping columns, coalesce to prefer fotmob values (fotmob first, then sofascore)
for col_name in overlapping_cols:
    gold_df = gold_df.withColumn(
        col_name,
        coalesce(col(f"fotmob.{col_name}"), col(f"{col_name}_sofascore"))
    ).drop(f"{col_name}_sofascore")

print(f"\nGold table will have {len(gold_df.columns)} columns")
print("\nSample of combined data:")
display(gold_df.limit(5))

After join: 6777 rows

Gold table will have 181 columns

Sample of combined data:


player_id,season,competition,accurate_long_balls,accurate_passes,aerials_won,aerials_won_percentage,assists,big_chances_created,blocked_scoring_attempt,chances_created,clean_sheets,clearances,cross_accuracy,defensive_actions,dispossessed,dribbled_past,dribbles,dribbles_success_rate,duels_won,duels_won_percentage,fouls_committed,fouls_won,goals,goals_conceded_while_on_pitch,headed_shots,interceptions,long_ball_accuracy,pass_accuracy,penalties_awarded,penalties_conceded,penalty_goals,possession_won_final_3rd,recoveries,red_cards,shots,shots_on_target,successful_crosses,tackles,touches,touches_in_opposition_box,xa,xg,xg_against_while_on_pitch,xg_excl_penalty,xgot,yellow_cards,accurate_long_balls_per90,accurate_passes_per90,aerials_won_per90,aerials_won_percentage_per90,assists_per90,big_chances_created_per90,blocked_scoring_attempt_per90,chances_created_per90,clean_sheets_per90,clearances_per90,cross_accuracy_per90,defensive_actions_per90,dispossessed_per90,dribbled_past_per90,dribbles_per90,dribbles_success_rate_per90,duels_won_per90,duels_won_percentage_per90,fouls_committed_per90,fouls_won_per90,goals_per90,goals_conceded_while_on_pitch_per90,headed_shots_per90,interceptions_per90,long_ball_accuracy_per90,pass_accuracy_per90,penalties_awarded_per90,penalties_conceded_per90,penalty_goals_per90,possession_won_final_3rd_per90,recoveries_per90,red_cards_per90,shots_per90,shots_on_target_per90,successful_crosses_per90,tackles_per90,touches_per90,touches_in_opposition_box_per90,xa_per90,xg_per90,xg_against_while_on_pitch_per90,xg_excl_penalty_per90,xgot_per90,yellow_cards_per90,accurate_long_balls_percentile,accurate_passes_percentile,aerials_won_percentile,aerials_won_percentage_percentile,assists_percentile,big_chances_created_percentile,blocked_scoring_attempt_percentile,chances_created_percentile,clean_sheets_percentile,clearances_percentile,cross_accuracy_percentile,defensive_actions_percentile,dispossessed_percentile,dribbled_past_percentile,dribbles_percentile,dribbles_success_rate_percentile,duels_won_percentile,duels_won_percentage_percentile,fouls_committed_percentile,fouls_won_percentile,goals_percentile,goals_conceded_while_on_pitch_percentile,headed_shots_percentile,interceptions_percentile,long_ball_accuracy_percentile,pass_accuracy_percentile,penalties_awarded_percentile,penalties_conceded_percentile,penalty_goals_percentile,possession_won_final_3rd_percentile,recoveries_percentile,red_cards_percentile,shots_percentile,shots_on_target_percentile,successful_crosses_percentile,tackles_percentile,touches_percentile,touches_in_opposition_box_percentile,xa_percentile,xg_percentile,xg_against_while_on_pitch_percentile,xg_excl_penalty_percentile,xgot_percentile,yellow_cards_percentile,accurate_long_balls_percentile_per90,accurate_passes_percentile_per90,aerials_won_percentile_per90,aerials_won_percentage_percentile_per90,assists_percentile_per90,big_chances_created_percentile_per90,blocked_scoring_attempt_percentile_per90,chances_created_percentile_per90,clean_sheets_percentile_per90,clearances_percentile_per90,cross_accuracy_percentile_per90,defensive_actions_percentile_per90,dispossessed_percentile_per90,dribbled_past_percentile_per90,dribbles_percentile_per90,dribbles_success_rate_percentile_per90,duels_won_percentile_per90,duels_won_percentage_percentile_per90,fouls_committed_percentile_per90,fouls_won_percentile_per90,goals_percentile_per90,goals_conceded_while_on_pitch_percentile_per90,headed_shots_percentile_per90,interceptions_percentile_per90,long_ball_accuracy_percentile_per90,pass_accuracy_percentile_per90,penalties_awarded_percentile_per90,penalties_conceded_percentile_per90,penalty_goals_percentile_per90,possession_won_final_3rd_percentile_per90,recoveries_percentile_per90,red_cards_percentile_per90,shots_percentile_per90,shots_on_target_percentile_per90,successful_crosses_percentile_per90,tackles_percentile_per90,touches_percentile_per90,touches_in_opposition_box_percentile_per90,xa_percentile_per90,xg_percenti

In [0]:
from pyspark.sql.functions import lit, when, current_timestamp

# Add metadata columns to track data source
gold_df = gold_df.withColumn(
    "data_source",
    when(col("fotmob.player_id").isNotNull() & col("sofascore.player_id").isNotNull(), lit("both"))
    .when(col("fotmob.player_id").isNotNull(), lit("fotmob_only"))
    .otherwise(lit("sofascore_only"))
).withColumn(
    "last_updated",
    current_timestamp()
)

# Show source distribution
print("Data source distribution:")
gold_df.groupBy("data_source").count().orderBy("data_source").show()

# Clean up the aliased join columns (remove fotmob. and sofascore. prefixes)
for col_name in join_keys:
    if f"fotmob.{col_name}" in gold_df.columns:
        gold_df = gold_df.withColumn(
            col_name,
            coalesce(col(f"fotmob.{col_name}"), col(f"sofascore.{col_name}"))
        ).drop(f"fotmob.{col_name}").drop(f"sofascore.{col_name}")

Data source distribution:
+--------------+-----+
|   data_source|count|
+--------------+-----+
|   fotmob_only| 4870|
|sofascore_only| 1907|
+--------------+-----+



In [0]:
# Check if gold table exists and inspect its schema
gold_table_name = "workspace.fotmob.player_stats_gold"

if spark.catalog.tableExists(gold_table_name):
    print(f"Gold table {gold_table_name} exists - checking schema...")
    
    # Get schema
    gold_table = spark.read.table(gold_table_name)
    existing_columns = set(gold_table.columns)
    expected_columns = set(gold_df.columns)
    
    print(f"\nExisting table: {len(existing_columns)} columns")
    print(f"Expected (gold_df): {len(expected_columns)} columns")
    
    # Compare schemas
    extra_in_table = existing_columns - expected_columns
    missing_from_table = expected_columns - existing_columns
    
    needs_drop = False
    drop_reason = ""
    
    if extra_in_table:
        needs_drop = True
        drop_reason = f"Table has {len(extra_in_table)} extra columns not in gold_df (e.g., {', '.join(list(extra_in_table)[:5])})"
        print(f"\nExtra columns in table: {sorted(list(extra_in_table)[:10])}")
    
    if missing_from_table:
        needs_drop = True
        if drop_reason:
            drop_reason += f" and missing {len(missing_from_table)} columns from gold_df"
        else:
            drop_reason = f"Table is missing {len(missing_from_table)} columns from gold_df"
        print(f"\nMissing columns from table: {sorted(list(missing_from_table)[:10])}")
    
    if needs_drop:
        print(f"\n {drop_reason}")
        print("\nDropping and recreating gold table...")
        spark.sql(f"DROP TABLE IF EXISTS {gold_table_name}")
        print(f" Dropped {gold_table_name} - it will be recreated in the next cell with the new schema")
    else:
        print("\n Schema is up to date, no changes needed")
else:
    print(f"Gold table {gold_table_name} does not exist yet - will be created in next cell")

Gold table workspace.fotmob.player_stats_gold exists - checking schema...

Existing table: 183 columns
Expected (gold_df): 183 columns

 Schema is up to date, no changes needed


In [0]:
from delta.tables import DeltaTable

# Save to gold table
gold_table_name = "workspace.fotmob.player_stats_gold"

# Count rows
row_count = gold_df.count()
print(f"Writing {row_count} rows to gold table...")

# If table doesn't exist, create it
if not spark.catalog.tableExists(gold_table_name):
    gold_df.write.format("delta").mode("overwrite").saveAsTable(gold_table_name)
    print(f"\n✓ Created gold table {gold_table_name} with {row_count} rows")
else:
    # Check if table has any invalid 2-digit seasons (should be 4-digit years)
    existing_table = spark.read.table(gold_table_name)
    invalid_seasons = existing_table.filter("season < 100").count()
    
    if invalid_seasons > 0:
        print(f"\n⚠ Found {invalid_seasons} rows with 2-digit seasons in existing table")
        print("  Dropping and recreating table with corrected 4-digit season values...")
        spark.sql(f"DROP TABLE IF EXISTS {gold_table_name}")
        gold_df.write.format("delta").mode("overwrite").saveAsTable(gold_table_name)
        print(f"\n✓ Recreated gold table {gold_table_name} with {row_count} rows")
    else:
        # Table exists with valid data - use MERGE to upsert
        delta_table = DeltaTable.forName(spark, gold_table_name)
        delta_table.alias("target").merge(
            gold_df.alias("source"),
            "target.player_id = source.player_id AND target.season = source.season AND target.competition = source.competition"
        ).whenMatchedUpdateAll(
        ).whenNotMatchedInsertAll(
        ).execute()
        print(f"\n✓ Merged {row_count} records into {gold_table_name}")

# Show final stats
final_count = spark.read.table(gold_table_name).count()
final_cols = len(spark.read.table(gold_table_name).columns)

print(f"\nFinal gold table stats:")
print(f"  - Total rows: {final_count}")
print(f"  - Total columns: {final_cols}")
print(f"  - Table name: {gold_table_name}")

# Show sample
print("\nSample from gold table:")
display(spark.read.table(gold_table_name).limit(5))

Writing 6777 rows to gold table...

✓ Merged 6777 records into workspace.fotmob.player_stats_gold

Final gold table stats:
  - Total rows: 6777
  - Total columns: 183
  - Table name: workspace.fotmob.player_stats_gold

Sample from gold table:


player_id,season,competition,accurate_long_balls,accurate_passes,aerials_won,aerials_won_percentage,assists,big_chances_created,blocked_scoring_attempt,chances_created,clean_sheets,clearances,cross_accuracy,defensive_actions,dispossessed,dribbled_past,dribbles,dribbles_success_rate,duels_won,duels_won_percentage,fouls_committed,fouls_won,goals,goals_conceded_while_on_pitch,headed_shots,interceptions,long_ball_accuracy,pass_accuracy,penalties_awarded,penalties_conceded,penalty_goals,possession_won_final_3rd,recoveries,red_cards,shots,shots_on_target,successful_crosses,tackles,touches,touches_in_opposition_box,xa,xg,xg_against_while_on_pitch,xg_excl_penalty,xgot,yellow_cards,accurate_long_balls_per90,accurate_passes_per90,aerials_won_per90,aerials_won_percentage_per90,assists_per90,big_chances_created_per90,blocked_scoring_attempt_per90,chances_created_per90,clean_sheets_per90,clearances_per90,cross_accuracy_per90,defensive_actions_per90,dispossessed_per90,dribbled_past_per90,dribbles_per90,dribbles_success_rate_per90,duels_won_per90,duels_won_percentage_per90,fouls_committed_per90,fouls_won_per90,goals_per90,goals_conceded_while_on_pitch_per90,headed_shots_per90,interceptions_per90,long_ball_accuracy_per90,pass_accuracy_per90,penalties_awarded_per90,penalties_conceded_per90,penalty_goals_per90,possession_won_final_3rd_per90,recoveries_per90,red_cards_per90,shots_per90,shots_on_target_per90,successful_crosses_per90,tackles_per90,touches_per90,touches_in_opposition_box_per90,xa_per90,xg_per90,xg_against_while_on_pitch_per90,xg_excl_penalty_per90,xgot_per90,yellow_cards_per90,accurate_long_balls_percentile,accurate_passes_percentile,aerials_won_percentile,aerials_won_percentage_percentile,assists_percentile,big_chances_created_percentile,blocked_scoring_attempt_percentile,chances_created_percentile,clean_sheets_percentile,clearances_percentile,cross_accuracy_percentile,defensive_actions_percentile,dispossessed_percentile,dribbled_past_percentile,dribbles_percentile,dribbles_success_rate_percentile,duels_won_percentile,duels_won_percentage_percentile,fouls_committed_percentile,fouls_won_percentile,goals_percentile,goals_conceded_while_on_pitch_percentile,headed_shots_percentile,interceptions_percentile,long_ball_accuracy_percentile,pass_accuracy_percentile,penalties_awarded_percentile,penalties_conceded_percentile,penalty_goals_percentile,possession_won_final_3rd_percentile,recoveries_percentile,red_cards_percentile,shots_percentile,shots_on_target_percentile,successful_crosses_percentile,tackles_percentile,touches_percentile,touches_in_opposition_box_percentile,xa_percentile,xg_percentile,xg_against_while_on_pitch_percentile,xg_excl_penalty_percentile,xgot_percentile,yellow_cards_percentile,accurate_long_balls_percentile_per90,accurate_passes_percentile_per90,aerials_won_percentile_per90,aerials_won_percentage_percentile_per90,assists_percentile_per90,big_chances_created_percentile_per90,blocked_scoring_attempt_percentile_per90,chances_created_percentile_per90,clean_sheets_percentile_per90,clearances_percentile_per90,cross_accuracy_percentile_per90,defensive_actions_percentile_per90,dispossessed_percentile_per90,dribbled_past_percentile_per90,dribbles_percentile_per90,dribbles_success_rate_percentile_per90,duels_won_percentile_per90,duels_won_percentage_percentile_per90,fouls_committed_percentile_per90,fouls_won_percentile_per90,goals_percentile_per90,goals_conceded_while_on_pitch_percentile_per90,headed_shots_percentile_per90,interceptions_percentile_per90,long_ball_accuracy_percentile_per90,pass_accuracy_percentile_per90,penalties_awarded_percentile_per90,penalties_conceded_percentile_per90,penalty_goals_percentile_per90,possession_won_final_3rd_percentile_per90,recoveries_percentile_per90,red_cards_percentile_per90,shots_percentile_per90,shots_on_target_percentile_per90,successful_crosses_percentile_per90,tackles_percentile_per90,touches_percentile_per90,touches_in_opposition_box_percentile_per90,xa_percentile_per90,xg_percenti